# Introduction

Let's start with the most obvious **problems** that can be seen by just looking at the database
and here are they:
#### LinkedIn
- [x] the `posted_since` column is relative to the collection date, not absolute
- [x] in the `seniority_level` column there's a value called "Not Applicable"
- [ ] Extract the `salary` **PAIN**
#### UpWork
- [x] There are empty strings in the `skills` column
- [ ] there are non **ASCII** charchters in the `description` column
- [x] A lot of columns have useless string components such as *"1978 <ins>hours worked</ins>"*
- [x] Money is represented using strings
- [x] Sometimes values are null and sometimes are strings indicating empty value.
#### Guru
- [x] the `earnings` and `feedback_percent` columns are represented as strings

# Setting up

In [9]:
%%capture
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime, timedelta, date
from transformers import pipeline
from typing import Optional
import sqlite3
import time
import ast
import re
import os

RAW_DB_URI = "file:./data/raw_database.db?mode=ro"
CLEAN_DB_PATH = "./data/clean_database.db"

In [10]:
with sqlite3.connect(RAW_DB_URI, uri=True) as con:
    linkedin_df = pd.read_sql_query("SELECT * FROM linkedin", con)
    upwork_df = pd.read_sql_query("SELECT * FROM upwork", con)
    guru_df = pd.read_sql_query("SELECT * FROM guru", con)

In [11]:
linkedin_df.sample(5)

,id,posting_title,location,posted_since,company_name,description,job_url,company_url,applicants,industries,employment_type,job_function,seniority_level,searched_country,searched_job_title
296,4296934176,Insights Analyst,"County Dublin, Ireland",1 day ago,LinkedIn,Company Description\nLinkedIn is the world’s l...,https://ie.linkedin.com/jobs/view/insights-ana...,https://www.linkedin.com/company/linkedin?trk=...,Over 200 applicants,"Technology, Information and Internet",Full-time,Sales,Associate,European Union,Data analyst
96,4293802529,Impiegato/a Data Entry – Inserimento Ordini,"Cattolica, Emilia-Romagna, Italy",6 days ago,Rhino online,Azienda attiva nella produzione e distribuzion...,https://it.linkedin.com/jobs/view/impiegato-a-...,https://www.linkedin.com/company/rhino-online?...,None,Advertising Services,Full-time,Other,Entry level,European Union,Data entry
823,4286271133,Marketing Data Analyst,"Plano, TX",2 days ago,At Home Group Inc.,Job Summary\n JOB DESCRIPTION \nAt Home Group ...,https://www.linkedin.com/jobs/view/marketing-d...,https://www.linkedin.com/company/athomestores?...,Over 200 applicants,Retail,Full-time,Information Technology,Mid-Senior level,United States,Data analyst
366,4293186093,Data Analyst,"Bucharest, Romania",4 days ago,Hays,We are supporting our client in strengthening ...,https://ro.linkedin.com/jobs/view/data-analyst...,https://uk.linkedin.com/company/hays?trk=publi...,None,IT Services and IT Consulting,Contract,Information Technology,Mid-Senior level,European Union,Data analyst
778,4296136090,"Senior Data Scientist, Reliability",San Francisco Bay Area,3 days ago,Block,"Block is one company built from many blocks, a...",https://www.linkedin.com/jobs/view/senior-data...,https://www.linkedin.com/company/joinblock?trk...,None,Financial Services,Full-time,Engineering and Information Technology,Mid-Senior level,United States,Data scientist


# Data cleaning

### LinkedIn

Fixing the `posted_since` column to use dates instead of days since the data was collected<br>
NOTE: it will still be an approximatation because linkedin doesn't specify actual posting date

In [12]:
linkedin_df["posted_since"].unique()

array(['5 days ago', '7 months ago', '3 weeks ago', '2 days ago',
       '2 weeks ago', '1 week ago', '6 days ago', '4 months ago',
       '3 days ago', '4 days ago', '1 month ago', '3 months ago',
       '4 weeks ago', '2 months ago', '5 months ago', '1 day ago',
       '13 hours ago', '18 hours ago', '22 hours ago', '23 hours ago',
       '21 hours ago', '4 hours ago', '14 hours ago', '12 hours ago',
       '17 hours ago', '1 hour ago', '3 hours ago', '7 hours ago',
       '15 hours ago', '16 hours ago', '5 hours ago', '2 hours ago',
       '8 hours ago', '10 hours ago', '9 hours ago'], dtype=object)

In [13]:
collection_time = datetime.fromtimestamp(os.path.getctime("./data/linkedin_jobs.csv"))

hour_pattern  = re.compile(r"^[0-9]+ hour")
day_pattern  = re.compile(r"^[0-9]+ day")
week_pattern  = re.compile(r"^[0-9]+ week")
month_pattern = re.compile(r"^[0-9]+ month")
year_pattern  = re.compile(r"^[0-9]+ year")

value_pattern = re.compile(r"^[0-9]+")

def parse_posted_since(collection_time: datetime, posted_since: str) -> date:
    used_pattern: re.Pattern = None
    
    for pattern in [hour_pattern, day_pattern, week_pattern,
                    month_pattern, year_pattern]:
        if re.match(pattern, posted_since):
            used_pattern = pattern
            break

    if used_pattern == None:
        raise ValueError("The `posted_since` param has invalid form that can't be parsed.")

    interval_value = int(value_pattern.search(posted_since).group())
    interval_unit: timedelta = datetime.hour
    
    if used_pattern == hour_pattern:
        interval_unit = timedelta(hours=1)

    elif used_pattern == day_pattern:
        interval_unit = timedelta(days=1)
        
    elif used_pattern == week_pattern:
        interval_unit = timedelta(weeks=1)
        
    elif used_pattern == month_pattern:
        interval_unit = timedelta(days=29.53)
        
    elif  used_pattern == year_pattern:
        interval_unit = timedelta(days=365.25)

    else:
        raise ValueError("The `posted_since` param has invalid form that can't be parsed.")

    interval = interval_value * interval_unit
    
    return datetime.date(collection_time - interval)

In [14]:
linkedin_df["posted_since"] = linkedin_df["posted_since"].apply(
    lambda x: parse_posted_since(collection_time, posted_since=x))

In [15]:
linkedin_df["posted_since"].sample(5)

923    2025-09-14
615    2025-09-10
708    2025-09-14
695    2025-09-12
610    2025-09-14
Name: posted_since, dtype: object

Replacing the "Not Applicable" value in the `seniority_level` column with None

In [16]:
linkedin_df["seniority_level"].unique()

array(['Not Applicable', 'Entry level', 'Mid-Senior level', 'Associate',
       'Internship', 'Director', 'Executive'], dtype=object)

In [17]:
linkedin_df["seniority_level"] = linkedin_df["seniority_level"].replace(
    "Not Applicable", None)

Now let's try to extract the salary using **hugging face**'s extractive Q/A model

In [40]:
# qa_model = pipeline("question-answering")
question = "What is the pay or salary range for the role?"
context = desc = linkedin_df["description"].sample(1).iloc[0]

print(qa_model(question = question, context = context)
print(desc.replace("$", "######").replace("€", "#####"))

{'score': 0.37302079796791077, 'start': 418, 'end': 430, 'answer': '$200 million'}
About The Company
About Cleo
At Cleo, we're not just building another fintech app. We're embarking on a mission to fundamentally change humanity's relationship with money. Imagine a world where everyone, regardless of background or income, has access to a hyper-intelligent financial advisor in their pocket. That's the future we're creating.
Cleo is a rare success story: a profitable, fast-growing unicorn with over ######200 million in ARR and growing over 2x year-over-year. This isn't just a job; it's a chance to join a team of brilliant, driven individuals who are passionate about making a real difference. We have an exceptionally high bar for talent, seeking individuals who are not only at the top of their field but also embody our culture of collaboration and positive impact.
If you’re driven by complex challenges that push your expertise, the chance to shape something truly transformative, and the po

Extracting skills from the job descriptions, the skills I am searching for are the ones<br>
who exist in the UpWork skills column instead of writing every skill I am searching for<br>
manualy.<br>

In [43]:
skills_set = set()
for skills in upwork_df["skills"].apply(ast.literal_eval):
    skills_set.update(skills)

In [ ]:
# def boyer_moore_search(T: str, P: str) -> bool:
    

### UpWork

Fixing the empty strings in the `skills` column

In [17]:
list(upwork_df["skills"].sample(1))

["['Data Engineering', 'Apache Spark', 'Python', 'JavaScript', 'ETL', 'SnapLogic', 'Make.com', 'Pentaho', '', '', '', '', '', '', '']"]

In [18]:
upwork_df["skills"] = upwork_df["skills"].apply(
    lambda list_: str(list(filter(lambda s: len(s) > 0, ast.literal_eval(list_))))
)

In [19]:
list(upwork_df["skills"].sample(1))

["['Dashboard', 'Data Cleaning', 'Data Entry', 'Microsoft Power BI', 'Looker Studio', 'Microsoft Excel', 'Excel Macros']"]

Now let's remove the clutter strings from some of the columns

In [20]:
upwork_df[["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]].sample(5)

,hours_worked,hourly_jobs_done,fixed_jobs_done
568,236 hours worked,10 hourly jobs,10 fixed price jobs
594,111 hours worked,15 hourly jobs,126 fixed price jobs
113,4634 hours worked,64 hourly jobs,31 fixed price jobs
527,570 hours worked,7 hourly jobs,3 fixed price jobs
326,701 hours worked,8 hourly jobs,3 fixed price jobs


In [21]:
def extract_value(s: str) -> int | None:
    value_pattern = re.compile("[0-9]+")

    if not(isinstance(s, str)):
        return None

    match = value_pattern.search(s)

    if not(match):
        return None

    return int(match.group())

for col in ["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]:
    upwork_df[col] = upwork_df[col].apply(extract_value)

In [22]:
upwork_df["hours_worked"].sample(5)

374      542.0
13     13029.0
260     2581.0
625     5949.0
132        NaN
Name: hours_worked, dtype: float64

Converting the money format from being a string into being a float for the `hour_rate` and `earnings`<br>
columns

In [23]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,None,$4.8
1,$2K+ earned,$3.5
2,$10K+ earned,$5
3,$100K+ earned,$5
4,None,$5


In [24]:
hour_rate_pattern = re.compile(r"\$[0-9.]+")
earnings_pattern = re.compile(r"\$[0-9]+")

def extract_earnings(s: str) -> int | None:
    if not(isinstance(s, str)):
        return None

    match = earnings_pattern.search(s)
    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000

    return int(match.group()[1:]) * magnitude

def extract_hour_rate(s: str) -> float | None:
    if not(isinstance(s, str)):
        return None

    match = hour_rate_pattern.search(s)

    if not(match):
        return None

    return float(match.group()[1:])

upwork_df["earnings"] = upwork_df["earnings"].apply(extract_earnings)
upwork_df["hour_rate"] = upwork_df["hour_rate"].apply(extract_hour_rate)

In [25]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,NaN,4.8
1,2000.0,3.5
2,10000.0,5.0
3,100000.0,5.0
4,NaN,5.0


### Guru

Let's fix the `feedback_percent` and `earnings` format & dtype

In [19]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
0,None,$0
1,100%,$28K
2,100%,$65K
3,100%,$489K
4,98.8%,$28K


In [20]:
def extract_feedback(s: str) -> float | None:
    pattern = re.compile(r"[0-9.]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    return float(match.group())

def extract_earnings(s: str) -> int | None:
    pattern = re.compile(r"\$[0-9,]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000
    
    earnings_str = match.group()[1:].replace(",", ".")

    return float(earnings_str) * magnitude

guru_df["feedback_percent"] = guru_df["feedback_percent"].apply(extract_feedback)
guru_df["earnings"] = guru_df["earnings"].apply(extract_earnings)

In [21]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
0,NaN,0.0
1,100.0,28000.0
2,100.0,65000.0
3,100.0,489000.0
4,98.8,28000.0


# Data storing

In [ ]:
with sqlite3.connect(CLEAN_DB_PATH) as con:
    linkedin_df.to_sql("linkedin", con, if_exists="fail", index=False)
    upwork_df.to_sql("upwork", con, if_exists="fail", index=False)
    guru_df.to_sql("guru", con, if_exists="fail", index=False)